# CE49X Lab 3: Where Should You Open a Gas Station in Istanbul?
## A Traffic-Based Site Selection Analysis

**Instructor:** Dr. Eyuphan Koc  
**Department of Civil Engineering, Bogazici University**  
**Semester:** Spring 2026

---

## Background

A fuel distribution company is planning to open **3 new gas stations** in Istanbul. They have hired you as a consulting engineer to identify the best locations based on **traffic patterns only**.

We provide a starter traffic dataset covering one week of hourly sensor readings across Istanbul (`istanbul_traffic_week.csv`).

Your job is to:
1. **Analyze traffic data** to understand where high-volume, low-speed (stop-and-go) traffic occurs — these are the locations where drivers are most likely to stop for fuel.
2. **Collect existing gas station data** for Istanbul to identify areas that are underserved.
3. **Propose 3 optimal locations** for new gas stations, supported by data and visualizations.

---
## Your Work Starts Here

### 1. Traffic Data — Source & Exploration

**Data Source:** I am using the provided dataset `istanbul_traffic_week.csv`, which contains 75,002 records from approximately 2,400 sensors across Istanbul for one week in October 2024. The data includes vehicle counts, speeds, and geographic coordinates.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load traffic data
df_traffic = pd.read_csv('../istanbul_traffic_week.csv')
df_traffic['DATE_TIME'] = pd.to_datetime(df_traffic['DATE_TIME'])

# Exploration
print(f"Dataset shape: {df_traffic.shape}")
print(df_traffic.head())

# Summary statistics per location
summary = df_traffic.groupby('GEOHASH').agg({
    'NUMBER_OF_VEHICLES': ['mean', 'sum'],
    'AVERAGE_SPEED': 'mean',
    'LATITUDE': 'first',
    'LONGITUDE': 'first'
}).reset_index()

summary.columns = ['GEOHASH', 'mean_hourly_vehicles', 'total_vehicles', 'mean_speed', 'lat', 'lon']

# Compute mean daily vehicle count (assuming 7 days of data)
summary['mean_daily_vehicles'] = summary['total_vehicles'] / 7

# Identify peak-hour vehicle count for each sensor
peak_counts = df_traffic.groupby('GEOHASH')['NUMBER_OF_VEHICLES'].max().reset_index()
peak_counts.columns = ['GEOHASH', 'peak_hour_vehicles']
summary = summary.merge(peak_counts, on='GEOHASH')

print("\nSummary Statistics (Head):")
print(summary.head())

# Top 20 highest-traffic locations
top_20 = summary.sort_values(by='total_vehicles', ascending=False).head(20)
print("\nTop 20 Highest-Traffic Locations:")
print(top_20[['GEOHASH', 'total_vehicles', 'mean_speed']])

In [ ]:
# Temporal patterns
df_traffic['hour'] = df_traffic['DATE_TIME'].dt.hour
df_traffic['day_of_week'] = df_traffic['DATE_TIME'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
sns.lineplot(data=df_traffic, x='hour', y='NUMBER_OF_VEHICLES', ci=None)
plt.title('Average Traffic Volume by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Avg Vehicles')
plt.grid(True)

plt.subplot(1, 2, 2)
sns.barplot(data=df_traffic, x='day_of_week', y='NUMBER_OF_VEHICLES', order=day_order)
plt.title('Average Traffic Volume by Day of Week')
plt.xlabel('Day')
plt.ylabel('Avg Vehicles')
plt.xticks(rotation=45)
plt.grid(True)

plt.tight_layout()
plt.show()

### 2. Traffic-Based Demand Scoring

**Demand Score Formula:**
$$\text{Demand Score} = 0.7 \times \text{Norm}(\text{Total Vehicles}) + 0.3 \times \text{Norm}(1 / \text{Average Speed})$$

**Justification:**
- **Vehicle Volume (70%):** Higher volume directly translates to more potential customers. It is the primary driver for gas station site selection.
- **Low Speed (30%):** Areas with lower average speeds indicate congestion or traffic signals. Drivers in slow or stop-and-go traffic are more likely to notice gas stations and have the time/opportunity to turn into one. We use the inverse of speed ($1/v$) to give higher scores to slower locations.
- **Normalization:** Both metrics are normalized to a 0-1 range to ensure they are comparable before weighted summation.

In [ ]:
def normalize(s):
    return (s - s.min()) / (s.max() - s.min())

summary['norm_volume'] = normalize(summary['total_vehicles'])
summary['norm_inv_speed'] = normalize(1 / summary['mean_speed'])
summary['demand_score'] = (summary['norm_volume'] * 0.7) + (summary['norm_inv_speed'] * 0.3)

ranked_locations = summary.sort_values(by='demand_score', ascending=False)
print("Top 10 Locations by Demand Score:")
print(ranked_locations[['GEOHASH', 'total_vehicles', 'mean_speed', 'demand_score']].head(10))

### 3. Existing Gas Station Data

**Data Source and Collection Method:**
I collected existing gas station data using the **Overpass API (OpenStreetMap)**. I queried for nodes and ways tagged with `amenity=fuel` within the bounding box of Istanbul. This method yielded **759 gas stations**, well exceeding the requirement of 200 stations.

In [ ]:
# Note: The data collection script was run externally to fetch 759 stations.
# Here we load the saved CSV and compute distances.
try:
    df_stations = pd.read_csv('../../existing_gas_stations.csv')
except:
    # Fallback if file not found (for demonstration in notebook environment)
    print("Loading locally collected station data...")
    df_stations = pd.read_csv('existing_gas_stations.csv')

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def get_min_dist(row):
    dists = haversine(row['lat'], row['lon'], df_stations['lat'], df_stations['lon'])
    return dists.min()

print("Computing distances to nearest stations...")
summary['dist_to_nearest'] = summary.apply(get_min_dist, axis=1)
print(f"Average distance to nearest station: {summary['dist_to_nearest'].mean():.2f} km")

### 4. Site Selection

To identify the optimal locations, I combined the **Demand Score** with the **Distance to Nearest Station**. A site is prioritized if it has high traffic demand and is currently underserved (far from existing stations).

**Final Score Formula:**
$$\text{Final Score} = \text{Demand Score} \times \text{Norm}(\text{Distance to Nearest})$$

In [ ]:
summary['norm_dist'] = normalize(summary['dist_to_nearest'])
summary['final_score'] = summary['demand_score'] * summary['norm_dist']

# Filter for reasonable city limits to avoid outliers in deep rural areas
city_mask = (summary['lat'] > 40.8) & (summary['lat'] < 41.3) & (summary['lon'] > 28.5) & (summary['lon'] < 29.5)
candidates = summary[city_mask].sort_values(by='final_score', ascending=False)

# Propose 3 Locations
proposals = candidates.head(3).copy()

# Manually lookup neighborhood names for the identified coordinates
proposals['Neighborhood'] = ['Sariyer (Maslak/Levent)', 'Sariyer (Northern Marmara Hwy)', 'Sariyer (Uskumruköy Area)']

print("Proposed Locations for New Gas Stations:")
display(proposals[['Neighborhood', 'lat', 'lon', 'demand_score', 'dist_to_nearest', 'final_score']])

### 5. Visualizations

In [ ]:
plt.figure(figsize=(12, 8))
plt.scatter(summary['lon'], summary['lat'], c=summary['demand_score'], cmap='YlOrRd', s=10, alpha=0.5)
plt.colorbar(label='Demand Score')
plt.scatter(proposals['lon'], proposals['lat'], c='blue', marker='*', s=200, label='Proposed Sites')
plt.title('Istanbul Gas Station Demand Heatmap')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
plt.scatter(df_stations['lon'], df_stations['lat'], c='gray', s=5, alpha=0.3, label='Existing Stations')
plt.scatter(proposals['lon'], proposals['lat'], c='red', marker='P', s=150, label='Proposed Sites')
plt.title('Existing vs. Proposed Gas Station Locations')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
for i, geohash in enumerate(proposals['GEOHASH']):
    site_data = df_traffic[df_traffic['GEOHASH'] == geohash].groupby('hour')['NUMBER_OF_VEHICLES'].mean()
    plt.plot(site_data.index, site_data.values, label=f"Site {i+1} ({proposals.iloc[i]['Neighborhood']})")

plt.title('Hourly Traffic Patterns at Proposed Locations')
plt.xlabel('Hour of Day')
plt.ylabel('Avg Vehicles')
plt.legend()
plt.grid(True)
plt.show()

### 6. Discussion

**Why these 3 locations?**
The three chosen locations represent a balance between massive traffic volume and lack of immediate competition. 
1. **Site 1 (Maslak/Levent area)** is in the heart of Istanbul's business district. Despite having stations nearby, the sheer volume of traffic (over 69,000 vehicles/week at the sensor) makes it highly profitable even with a 2.1km gap to the next station.
2. **Site 2 and 3** are located in the northern expansion areas of Istanbul, likely near the Northern Marmara Highway. These sites show significant vehicle counts (30k+) but are much further from existing infrastructure (over 5-7km), representing classic "underserved" highway corridors where drivers may run low on fuel without many options.

**Limitations:**
A traffic-only analysis ignores several critical real-world factors. First, **land cost and availability** in central areas like Maslak could be prohibitively high, making a gas station economically unviable. Second, **zoning regulations** may prohibit gas stations in certain high-traffic areas. Third, the **direction of traffic** matters; a station is only useful if it is accessible from the high-volume side of the road. Finally, the **type of vehicles** (trucks vs. passenger cars) significantly affects fuel demand but is not captured in this dataset.

**Additional Data:**
If I had access to **land use/zoning maps** and **land prices**, I could filter out locations where building is impossible or too expensive. Additionally, **vehicle type classification** (distinguishing heavy trucks from commuter cars) would allow for a much more accurate estimation of actual fuel demand, as trucks have much larger tanks and different refueling frequencies.

---

### Questions?

**Dr. Eyuphan Koc**  
eyuphan.koc@bogazici.edu.tr